# Novae on IMC: graph-foundation-model niche features for cell typing, and cohort alignment (J↔D)

Pre-registered in `future_plan.md` §18.9 (2026-11-04), before any code was run.

**Question.** Novae (Blampey 2025, *Nat. Methods*) is a self-supervised graph attention network (SwAV + optimal transport). It embeds each cell's 2-hop neighbourhood and corrects batch effects between slides without Harmony. Two questions:
- Do its niche embeddings add to the best cell-typing model (LightGBM on the cell table plus neighbour features), above all across cohorts?
- Does its alignment mix the two cohorts without merging cell types?

**Pretrained weights are not used.** They are RNA-only. Novae is trained from scratch on IMC, as the paper does for its proteomics example.

**Inputs:** the same 15 values as `cell-type-graph`: the 11 shared markers, H3, DNA1, DNA2 and log area, with arcsinh(x/5) for the channels.
- Novae z-scores each input within its "slide" itself.
- Its 20-gene minimum is lowered to 10.
- Gene embeddings: size 14, PCA-initialised.

**Graph:** Delaunay within each image, edges ≤ 40 µm (1 px = 1 µm).

**Novae training:** per replicate, on the train + val patients of **both** cohorts, without labels; test patients are never seen.
- This is unsupervised domain adaptation: transfer arms see unlabelled target-cohort training cells. The table baselines see only the target cohort's per-marker scaling statistics.

**Variants** (a Novae "slide" is the unit it z-scores and aligns):
- `[cohort]` (primary): one slide per cohort, so per-cohort z-scores and an optimal-transport queue across the two cohorts;
- `[img]`: one slide per image, Novae's default.

**Outputs per cell:**
- the 64-d latent;
- domains at level 10;
- for `[cohort]` only, the latent after `batch_effect_correction` (`-corr`).

**Arms** (LightGBM, as in `cell-type-graph`: same splits, seeds, N_FIT, class weights and labels):
- coarse labels: Table + nbr (reference); Novae `[cohort]` alone; Table + nbr + Novae `[cohort]`; Table + nbr + Novae `[cohort]`-corr; Table + nbr + Novae `[img]`;
- fine labels (D→D, J→J): Table + nbr and Table + nbr + Novae `[cohort]` only.

**Scores:** macro-F1 (primary), balanced accuracy and per-class F1, with 1000× patient-bootstrap CIs paired against Table + nbr, over 3 replicates.
- Table + nbr should reproduce `cell-type-graph` exactly; it is checked as a sanity row.

**Cohort mixing:**
- iLISI over cohort (1–2) and cLISI over the 6 coarse labels, perplexity 30;
- 5 draws of 5,000 labelled test cells per cohort;
- spaces: Table, Table + nbr, Novae `[cohort]`, `-corr` and `[img]`, each z-scored over the pooled draw.

**Decision rules (fixed in §18.9):**
- **H-N1:** Table + nbr + Novae `[cohort]` beats Table + nbr on coarse macro-F1, with paired CI > 0 in all 3 replicates and a mean gain ≥ +0.01, in J→D or D→J.
  - It must also not lose (CI < 0 in ≥ 2 replicates) in the other direction or within a cohort.
- **H-N2:** `[cohort]`-corr has higher median iLISI than Table + nbr in every draw, and its mean cLISI rises by < 0.25.

A time guard skips whole units that would pass ~11 h, so the kernel still ends COMPLETE.

In [ ]:
import os, sys, glob, json, time, warnings, gc, subprocess, logging
SMOKE = os.environ.get("SMOKE") == "1"

from importlib.metadata import version, PackageNotFoundError
PINS = []                                         # keep Kaggle's preinstalled core stack: a numpy upgrade under a live kernel breaks imports
for _p in ("numpy", "scipy", "pandas", "numba", "llvmlite", "pyarrow", "torch", "scikit-learn", "matplotlib", "h5py"):
    try:
        PINS.append(f"{_p}=={version(_p)}")
    except PackageNotFoundError:
        pass
open("/tmp/pip-constraints.txt", "w").write("\n".join(PINS) + "\n")

def pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--progress-bar", "off",
                    "-c", "/tmp/pip-constraints.txt", *args], check=True)

try:
    import novae
except ImportError:
    # novae pins pyarrow>=24: install it without dependencies and add the rest, so Kaggle's pyarrow/pandas stay as they are
    pip("--no-deps", "novae==1.1.2")
    pip("anndata>=0.11.0", "fast-array-utils>=1.3.1,<1.4.0", "huggingface-hub>=0.32.0", "igraph>=0.11.8",
        "lightning>=2.2.1", "safetensors>=0.4.3", "scanpy>=1.9.8", "torch-geometric>=2.5.2")
    import novae
try:
    import lightgbm as lgb
except ImportError:
    pip("lightgbm")
    import lightgbm as lgb
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
import anndata as ad
import torch
import lightning as L
from novae._constants import Nums
Nums.MIN_GENES = 10                               # 15 inputs; Novae's default minimum is 20 genes
if SMOKE:
    Nums.MIN_DATASET_LENGTH = 5_000               # cells per Novae epoch (default 50k)
warnings.filterwarnings("ignore")
for name in ("lightning", "lightning.pytorch", "lightning.fabric"):
    logging.getLogger(name).setLevel(logging.WARNING)
print("novae", novae.__version__, "| anndata", ad.__version__, "| torch", torch.__version__, "| lightning", L.__version__)

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SEED = 20261013                                   # same seed and split rules as cell-type-graph / train-test-grid
N_BOOT = 100 if SMOKE else 1000
MIN_CELLS = 200
COF = 5.0
N_FIT = 20_000 if SMOKE else 300_000              # labelled training cells per cohort for LightGBM (as cell-type-graph)
LGB = dict(n_estimators=30 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
BUDGET_MIN = float(os.environ.get("BUDGET_MIN", 660))
REPS = [0] if SMOKE else [int(r) for r in os.environ.get("REPS", "0,1,2").split(",")]
SMOKE_IMAGES = 40                                 # per cohort, every k-th image
dev = "cuda" if torch.cuda.is_available() else "cpu"
NW = 4 if dev == "cuda" else 0                    # Novae: dataloader workers are slow on CPU

NOVAE_INIT = dict(embedding_size=14, output_size=64, num_prototypes=512)   # embedding < 15 inputs, so PCA init runs
NOVAE_FIT = dict(max_epochs=3 if SMOKE else 30, lr=5e-4, patience=4, min_delta=0.05)  # > 2 warm-up epochs of Novae's LR schedule
RADIUS = 40.0                                     # µm, Delaunay edges kept
DOMAIN_LEVEL = 10
VARIANTS = ["cohort", "img"]
LISI_PERP = 30
LISI_N = 1000 if SMOKE else 5000                  # labelled test cells per cohort per draw
LISI_DRAWS = 2 if SMOKE else 5

NUC = ["histone_h3", "dna1", "dna2"]
MARKERS = ["c_erb_b_2_her2", "cd20", "cd3", "cd45", "cd68", "cytokeratin_5", "cytokeratin_8_18",
           "estrogen_receptor_alpha", "ki_67", "pan_cytokeratin", "sma"]
FEATS = NUC + MARKERS                             # + log area
VARS = FEATS + ["log_area"]
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}  # name -> Danenberg cell-table column

COARSE = ["epithelial", "T cell", "B cell", "myeloid", "endothelial", "stroma"]
D_COARSE = {"CD8^{+} T cells": "T cell", "CD4^{+} T cells": "T cell", "T_{Reg} & T_{Ex}": "T cell",
            "B cells": "B cell",
            "Macrophages": "myeloid", "Granulocytes": "myeloid", "Macrophages & granulocytes": "myeloid",
            "Endothelial": "endothelial",
            "Fibroblasts": "stroma", "Fibroblasts FSP1^{+}": "stroma", "Myofibroblasts": "stroma",
            "Myofibroblasts PDPN^{+}": "stroma"}
# Jackson by metacluster id. 1 and 2 by marker evidence (published names swapped); 3 and 5 unlabelled.
J_COARSE = {1: "T cell", 2: "B cell", 4: "myeloid", 6: "myeloid", 7: "endothelial",
            **{m: "stroma" for m in range(8, 14)}, **{m: "epithelial" for m in range(14, 28)}}
R_CAP = 32
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

## Cell tables, inputs and labels (as `cell-type-graph`)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "Danenberg" if "metabric_id" in pq.read_schema(p).names else "Jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p))
P["Danenberg"]["split"] = near("danenberg_patient_split.csv", P["Danenberg"]["cells"])
print(json.dumps(P, indent=1))

def load(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "Danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    keep = n_per.index[n_per >= MIN_CELLS]
    if SMOKE:
        keep = sorted(keep)[::max(1, len(keep) // SMOKE_IMAGES)]
    cells = cells[cells.image_name.isin(keep)]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    n = len(cells)
    img, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img)) + 1, n]
    loc = pd.Series(np.arange(n), index=pd.MultiIndex.from_arrays([cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    CON = sp.coo_matrix((np.ones(len(ca)), (ca, cb)), shape=(n, n)).tocsr()
    CON = ((CON + CON.T) > 0).astype(np.float64).tocsr()
    cols = [D_TABLE.get(c, c) for c in FEATS] if key == "Danenberg" else FEATS
    X = np.c_[np.arcsinh(cells[cols].values.astype(np.float64) / COF),
              np.log(np.maximum(cells["area"].values, 1))].astype(np.float32)
    phen = cells.cell_phenotype.astype(str).values
    fine = sorted(set(phen))
    y_fine = pd.Series(phen).map({p: i for i, p in enumerate(fine)}).values.astype(np.int64)
    if key == "Danenberg":
        epi = cells.groupby(cells.cell_phenotype.astype(str)).is_epithelial.mean()
        cmap = {p: ("epithelial" if epi[p] > 0.5 else D_COARSE.get(p)) for p in fine}
    else:
        mc = cells.groupby(cells.cell_phenotype.astype(str)).metacluster.first()
        cmap = {p: J_COARSE.get(int(mc[p])) for p in fine}
    coarse_of = np.array([COARSE.index(cmap[p]) if cmap[p] else -1 for p in fine], np.int64)
    return dict(key=key, n=n, starts=starts, n_img=len(img_names), img_names=np.asarray(img_names),
                cell_img=np.asarray(img_names)[img], obj=cells.object_number.values,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                CON=CON, X=X, fine=fine, cmap=cmap, y={"fine": y_fine, "coarse": coarse_of[y_fine]})

CO = {k: load(k) for k in ("Jackson", "Danenberg")}
CJ, CD = CO["Jackson"], CO["Danenberg"]
split0 = pd.read_csv(P["Danenberg"]["split"]).set_index("metabric_id").split
CD["split0"] = pd.Series(CD["patient"]).map(split0).fillna("none").values
for key, C in CO.items():
    print(f"{key}: {C['n']:,} cells, {C['n_img']} images, {len(set(C['patient']))} patients, {len(C['fine'])} fine labels; "
          f"coarse {pd.Series(C['y']['coarse']).map(dict(enumerate(COARSE))).fillna('unlabelled').value_counts().to_dict()}")

def harm(X):
    med = np.nanmedian(X, 0)
    q05, q95 = np.nanpercentile(X, [5, 95], 0)
    sc = np.where(q95 - q05 > 1e-3, q95 - q05, np.nanstd(X, 0) + 1e-6)
    return ((X - med) / sc).astype(np.float32)

for C in CO.values():
    C["XS"] = {"cohort": np.nan_to_num(harm(C["X"]))}
tick("tables loaded")

## Neighbour features (as `cell-type-graph`) and the Novae graph

In [ ]:
def adjacency(C, kind, par):
    rows, cols = [], []
    for a, b in zip(C["starts"][:-1], C["starts"][1:]):
        xy = C["xy"][a:b]
        k = min(par if kind == "knn" else R_CAP, b - a - 1)
        if kind == "knn":
            d, nb = cKDTree(xy).query(xy, k=k + 1)
        else:
            d, nb = cKDTree(xy).query(xy, k=k + 1, distance_upper_bound=par)
        d, nb = d[:, 1:], nb[:, 1:]
        ok = np.isfinite(d)
        rows.append(np.repeat(np.arange(a, b), k)[ok.ravel()])
        cols.append(nb[ok] + a)
    rows, cols = np.concatenate(rows), np.concatenate(cols)
    A = sp.coo_matrix((np.ones(len(rows), np.float32), (rows, cols)), shape=(C["n"], C["n"])).tocsr()
    A = ((A + A.T) > 0).astype(np.float32).tocsr()
    A.sort_indices()
    return A

def nbr_features(C, A, n_r30):
    X = C["XS"]["cohort"]
    deg = np.diff(A.indptr).astype(np.float32)[:, None]
    mean = (A @ X) / deg
    std = np.sqrt(np.maximum((A @ (X * X)) / deg - mean * mean, 0))
    mx = np.maximum.reduceat(X[A.indices], A.indptr[:-1], axis=0)
    hop2 = (A @ mean) / deg
    dst = np.repeat(np.arange(C["n"]), np.diff(A.indptr))
    dist = np.bincount(dst, np.linalg.norm(C["xy"][A.indices] - C["xy"][dst], axis=1), C["n"]) / deg.ravel()
    n_con = np.diff(C["CON"].indptr)
    return np.c_[X, mean, mx, std, hop2, np.log1p(n_r30), np.log1p(n_con), dist / 30.0].astype(np.float32)

for key, C in CO.items():
    A10 = adjacency(C, "knn", 10)
    A30 = adjacency(C, "radius", 30.0)
    C["XS"]["nbr"] = nbr_features(C, A10, np.diff(A30.indptr))
    del A10, A30
    gc.collect()
tick("neighbour features")

AD, GSTATS = {}, {}
for key, C in CO.items():
    a = ad.AnnData(X=np.nan_to_num(C["X"]).astype(np.float32))
    a.obs_names = [f"{key[0]}{i}" for i in range(C["n"])]
    a.var_names = VARS
    a.obs["image"] = pd.Categorical(pd.Series(C["cell_img"]).map(lambda s: f"{key[0]}|{s}").values)
    a.obsm["spatial"] = C["xy"]
    novae.spatial_neighbors(a, slide_key="image", radius=RADIUS, verbose=False)
    M = a.obsp["spatial_distances"].copy()
    M.eliminate_zeros()
    deg = np.diff(M.indptr)
    GSTATS[key] = dict(mean_degree=float(deg.mean()), isolated_fraction=float((deg == 0).mean()),
                       median_edge_um=float(np.median(M.data)))
    AD[key] = a
    del M
json.dump(GSTATS, open(f"{OUT}/graph_stats.json", "w"), indent=1)
print(pd.DataFrame(GSTATS).T.round(3).to_string())
tick("Delaunay graphs")

## Novae: fit on train + val cells of both cohorts, embed every cell

In [ ]:
class LossLog(L.Callback):
    def __init__(s):
        s.rows = []
    def on_train_epoch_end(s, trainer, pl_module):
        r = {k: float(v) for k, v in trainer.callback_metrics.items()}
        r["epoch"] = int(trainer.current_epoch)
        s.rows.append(r)

def set_slides(variant):
    for key, a in AD.items():
        sid = np.full(a.n_obs, key) if variant == "cohort" else a.obs["image"].astype(str).values
        a.obs["novae_sid"] = pd.Categorical(sid)

def run_novae(variant, IMG, seed):
    t0 = time.time()
    L.seed_everything(seed, workers=True)
    set_slides(variant)
    train = [AD[k][cells_of(CO[k], np.r_[IMG[k]["train"], IMG[k]["val"]])].copy() for k in ("Jackson", "Danenberg")]
    n_train = sum(t.n_obs for t in train)
    model = novae.Novae(train, **NOVAE_INIT)
    cb = LossLog()
    model.fit(accelerator=dev, num_workers=NW, callbacks=[cb], enable_progress_bar=False, devices=1, **NOVAE_FIT)
    t_fit = time.time() - t0
    del train
    gc.collect()
    adatas = [AD["Jackson"], AD["Danenberg"]]
    model.compute_representations(adatas, accelerator=dev, num_workers=NW)
    dom = model.assign_domains(adatas, level=DOMAIN_LEVEL)
    if variant == "cohort":
        model.batch_effect_correction(adatas, dom)
    res = {}
    for k, a in AD.items():
        r = dict(lat=np.asarray(a.obsm["novae_latent"], np.float32), valid=a.obs["neighborhood_valid"].values.astype(bool),
                 domain=a.obs[dom].astype(str).values)
        if variant == "cohort":
            r["corr"] = np.asarray(a.obsm["novae_latent_corrected"], np.float32)
        res[k] = r
    log = dict(variant=variant, n_train_cells=n_train, epochs=len(cb.rows), fit_min=t_fit / 60,
               total_min=(time.time() - t0) / 60,
               final_loss=cb.rows[-1].get("train/loss_epoch", np.nan) if cb.rows else np.nan,
               **{f"valid_{k}": float(r["valid"].mean()) for k, r in res.items()},
               **{f"domains_{k}": int(len(set(r["domain"]) - {"nan"})) for k, r in res.items()})
    del model
    if dev == "cuda":
        torch.cuda.empty_cache()
    return res, [dict(variant=variant, **r) for r in cb.rows], log

## LightGBM arms and scoring (patient bootstrap over the test patients)

In [ ]:
ARMS = {"Table + nbr": ("nbr",),
        "Novae [cohort]": ("nov_cohort",),
        "Table + nbr + Novae [cohort]": ("nbr", "nov_cohort"),
        "Table + nbr + Novae [cohort]-corr": ("nbr", "nov_cohort_corr"),
        "Table + nbr + Novae [img]": ("nbr", "nov_img")}
FINE_ARMS = ["Table + nbr", "Table + nbr + Novae [cohort]"]
BASE = "Table + nbr"
LISI_SPACES = {"Table": ("cohort",), "Table + nbr": ("nbr",), "Novae [cohort]": ("nov_cohort",),
               "Novae [cohort]-corr": ("nov_cohort_corr",), "Novae [img]": ("nov_img",)}

def feats(C, fs, idx):
    return np.hstack([C["XS"][f][idx] for f in fs])

def class_weights(y, ncls):
    y = y[y >= 0]
    cnt = np.bincount(y, minlength=ncls).astype(np.float64)
    w = np.where(cnt > 0, 1 / np.sqrt(np.maximum(cnt, 1)), 0.0)
    return w / w[y].mean()

def cells_of(C, imgs):
    st = C["starts"]
    return np.concatenate([np.arange(st[i], st[i + 1]) for i in imgs]) if len(imgs) else np.zeros(0, np.int64)

def fit_table(fs, src, task, ncls, IMG, seed):
    rs = np.random.default_rng(seed)
    Xs, ys = [], []
    for key in (["Jackson", "Danenberg"] if src == "Both" else [src]):
        C = CO[key]
        idx = cells_of(C, IMG[key]["train"])
        idx = idx[C["y"][task][idx] >= 0]
        idx = np.sort(rs.choice(idx, min(N_FIT, len(idx)), replace=False))
        Xs.append(feats(C, fs, idx))
        ys.append(C["y"][task][idx])
    X, y = np.vstack(Xs), np.concatenate(ys)
    mdl = lgb.LGBMClassifier(random_state=seed, **LGB).fit(X, y, sample_weight=class_weights(y, ncls)[y])
    return mdl, len(y)

def predict_table(mdl, fs, C, imgs, task):
    out = np.full(C["n"], -1, np.int64)
    node = cells_of(C, imgs)
    node = node[C["y"][task][node] >= 0]
    for s in range(0, len(node), 200_000):
        nd = node[s:s + 200_000]
        out[nd] = mdl.classes_[mdl.predict_proba(feats(C, fs, nd)).argmax(1)]
    return out

def scorer(C, imgs, task, ncls, seed):
    node = cells_of(C, imgs)
    node = node[C["y"][task][node] >= 0]
    pats, pc = np.unique(C["patient"][node], return_inverse=True)
    W = np.random.default_rng(seed).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
    y = C["y"][task][node]
    return dict(node=node, y=y, pc=pc, npat=len(pats), W=W, ncls=ncls, present=np.bincount(y, minlength=ncls) > 0)

def metrics(M, ncls, present):
    M = M.reshape(-1, ncls, ncls)
    tp = np.einsum("bii->bi", M)
    sup, prd = M.sum(2), M.sum(1)
    with np.errstate(invalid="ignore", divide="ignore"):
        f1 = np.where(sup + prd > 0, 2 * tp / (sup + prd), 0.0)
        rec = np.where(sup > 0, tp / sup, np.nan)
        out = dict(macro_f1=f1[:, present].mean(1), bal_acc=np.nanmean(rec[:, present], 1), acc=tp.sum(1) / M.sum((1, 2)))
    return out, f1

def score(S, pred_full):
    p = pred_full[S["node"]]
    assert (p >= 0).all(), "missing predictions"
    k = S["ncls"]
    cm = np.bincount((S["pc"] * k + S["y"]) * k + p, minlength=S["npat"] * k * k).reshape(S["npat"], k * k).astype(np.float64)
    tot = cm.sum(0, keepdims=True)
    pt, f1pt = metrics(tot, k, S["present"])
    bt, f1bt = metrics(S["W"] @ cm, k, S["present"])
    return {m: float(v[0]) for m, v in pt.items()}, bt, f1pt[0], f1bt, tot.reshape(k, k)

def ci(v):
    v = np.asarray(v)[np.isfinite(v)]
    return float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))

def labels_of(task, test):
    return COARSE if task == "coarse" else CO[test]["fine"]

## LISI (harmonypy's perplexity calibration, ported as in `lisi-imc`)

In [ ]:
def knn(V, k=LISI_PERP * 3):
    """k nearest neighbours including self, as in harmonypy; self is dropped afterwards."""
    Ds, Is = [], []
    for s in range(0, len(V), 2000):
        D = cdist(V[s:s + 2000], V)
        idx = np.argpartition(D, k - 1, axis=1)[:, :k]
        d = np.take_along_axis(D, idx, 1)
        o = np.argsort(d, axis=1, kind="stable")
        Is.append(np.take_along_axis(idx, o, 1))
        Ds.append(np.take_along_axis(d, o, 1))
    D, I = np.vstack(Ds), np.vstack(Is)
    return D[:, 1:], I[:, 1:]

def calibrate(D, perplexity=LISI_PERP, tol=1e-5, n_tries=50):
    n = len(D)
    logU = np.log(perplexity)
    beta, bmin, bmax = np.ones(n), np.full(n, -np.inf), np.full(n, np.inf)
    def hp(b):
        P = np.exp(-D * b[:, None])
        s = P.sum(1)
        with np.errstate(divide="ignore", invalid="ignore"):
            H = np.where(s > 0, np.log(s) + b * (D * P).sum(1) / s, 0.0)
            P = np.where(s[:, None] > 0, P / s[:, None], 0.0)
        return H, P
    H, P = hp(beta)
    for _ in range(n_tries):
        act = np.abs(H - logU) >= tol
        if not act.any():
            break
        pos, neg = act & (H - logU > 0), act & ~(H - logU > 0)
        bmin[pos] = beta[pos]
        beta[pos] = np.where(np.isfinite(bmax[pos]), (beta[pos] + bmax[pos]) / 2, beta[pos] * 2)
        bmax[neg] = beta[neg]
        beta[neg] = np.where(np.isfinite(bmin[neg]), (beta[neg] + bmin[neg]) / 2, beta[neg] / 2)
        Hn, Pn = hp(beta)
        H[act], P[act] = Hn[act], Pn[act]
    return H, P

def lisi_from(H, P, I, codes, n_cat):
    n, k = I.shape
    lab = codes[I]
    w = np.bincount((np.arange(n)[:, None] * n_cat + lab).ravel(), P.ravel(), n * n_cat).reshape(n, n_cat)
    simpson = (w ** 2).sum(1) + np.where(H == 0, -1.0, 0.0)
    return 1.0 / simpson

r = np.random.default_rng(0)
Xs = np.r_[r.normal(0, 1, (500, 6)), r.normal(8, 1, (500, 6))]
D_, I_ = knn(Xs); H_, P_ = calibrate(D_)
sep = np.median(lisi_from(H_, P_, I_, np.r_[np.zeros(500, int), np.ones(500, int)], 2))
mix = np.median(lisi_from(H_, P_, I_, r.integers(0, 2, 1000), 2))
print(f"LISI check: separated {sep:.3f} (expect ~1), random {mix:.3f} (expect ~2)")
assert sep < 1.05 and mix > 1.8

def lisi_rows(REP, IMG):
    rows = []
    rs = np.random.default_rng(SEED + 41 + REP)
    pools = {}
    for k, C in CO.items():
        p = cells_of(C, IMG[k]["test"])
        pools[k] = p[(C["y"]["coarse"][p] >= 0) & C["nov_valid"][p]]
    for d in range(LISI_DRAWS):
        pick = {k: rs.choice(p, min(LISI_N, len(p)), replace=False) for k, p in pools.items()}
        coh = np.r_[np.zeros(len(pick["Jackson"]), np.int64), np.ones(len(pick["Danenberg"]), np.int64)]
        lab = np.r_[CJ["y"]["coarse"][pick["Jackson"]], CD["y"]["coarse"][pick["Danenberg"]]]
        for space, fs in LISI_SPACES.items():
            V = np.vstack([feats(CO[k], fs, pick[k]) for k in ("Jackson", "Danenberg")]).astype(np.float64)
            V = (V - V.mean(0)) / (V.std(0) + 1e-9)
            D, I = knn(V)
            H, Pm = calibrate(D)
            il, cl = lisi_from(H, Pm, I, coh, 2), lisi_from(H, Pm, I, lab, len(COARSE))
            rows.append(dict(rep=REP, draw=d, space=space, n=len(V), ilisi_median=float(np.median(il)),
                             ilisi_mean=float(il.mean()), clisi_median=float(np.median(cl)), clisi_mean=float(cl.mean())))
    return rows

## Replicates: Novae fits, then LightGBM arms × tasks × training sets

In [ ]:
UNITS = [("coarse", "Jackson"), ("coarse", "Danenberg"), ("coarse", "Both"), ("fine", "Danenberg"), ("fine", "Jackson")]
EST, SKIPPED, NOVAE_LOG, NOVAE_LOSS, LISI = {}, [], [], [], []
D_TRVA_PAT = np.unique(CD["patient"][CD["split0"] != "test"])

def split_images(C, cell_split):
    s = cell_split[C["starts"][:-1]]
    return {k: np.flatnonzero(s == k) for k in ("train", "val", "test")}

def make_splits(REP):
    jp, jcode = np.unique(CJ["patient"], return_inverse=True)
    u = np.random.default_rng(SEED + 11 + REP).random(len(jp))
    J_SPLIT = np.where(u < 0.7, "train", np.where(u < 0.8, "val", "test"))[jcode]
    uv = np.random.default_rng(SEED + 21 + REP).random(len(D_TRVA_PAT))
    dval = set(D_TRVA_PAT[uv < 0.125])
    D_SPLIT = np.where(CD["split0"] == "test", "test", np.where(pd.Series(CD["patient"]).isin(dval).values, "val", "train"))
    IMG = {"Jackson": split_images(CJ, J_SPLIT), "Danenberg": split_images(CD, D_SPLIT)}
    for c in ("Jackson", "Danenberg"):
        print(c, {s: (len(v), len(set(CO[c]["patient"][cells_of(CO[c], v)]))) for s, v in IMG[c].items()}, "(images, patients)")
    return IMG

def save_latents(REP, NOV):
    os.makedirs(f"{OUT}/latents", exist_ok=True)
    for k, C in CO.items():
        np.savez_compressed(f"{OUT}/latents/novae_rep{REP}_{k}.npz", image_name=C["cell_img"], object_number=C["obj"],
                            cohort=NOV["cohort"][k]["lat"].astype(np.float16), cohort_corr=NOV["cohort"][k]["corr"].astype(np.float16),
                            img=NOV["img"][k]["lat"].astype(np.float16), valid_cohort=NOV["cohort"][k]["valid"],
                            valid_img=NOV["img"][k]["valid"])
        pd.DataFrame(dict(image_name=C["cell_img"], object_number=C["obj"], domain_cohort=NOV["cohort"][k]["domain"],
                          domain_img=NOV["img"][k]["domain"])).to_parquet(f"{OUT}/latents/novae_domains_rep{REP}_{k}.parquet")

for REP in REPS:
    el = (time.time() - T0) / 60
    proj = EST.get("rep", 200.0)
    if REP != REPS[0] and el + proj > BUDGET_MIN:
        print(f"!!! skipping replicate {REP}: {el:.0f} + {proj:.0f} min > {BUDGET_MIN:.0f}")
        SKIPPED.append(("rep", REP))
        continue
    t_rep = time.time()
    tick(f"replicate {REP}")
    IMG = make_splits(REP)
    NOV = {}
    for variant in VARIANTS:
        res, loss, log = run_novae(variant, IMG, SEED + 51 + REP)
        NOV[variant] = res
        NOVAE_LOSS += [dict(rep=REP, **r) for r in loss]
        NOVAE_LOG.append(dict(rep=REP, **log))
        print(json.dumps(NOVAE_LOG[-1]))
        tick(f"rep {REP} Novae [{variant}] done")
    for k, C in CO.items():
        C["XS"]["nov_cohort"] = NOV["cohort"][k]["lat"]
        C["XS"]["nov_cohort_corr"] = NOV["cohort"][k]["corr"]
        C["XS"]["nov_img"] = NOV["img"][k]["lat"]
        C["nov_valid"] = NOV["cohort"][k]["valid"] & NOV["img"][k]["valid"]
    if REP == REPS[0]:
        save_latents(REP, NOV)
    pd.DataFrame(NOVAE_LOG).to_csv(f"{OUT}/novae_log.csv", index=False)
    pd.DataFrame(NOVAE_LOSS).to_csv(f"{OUT}/novae_loss.csv", index=False)
    del NOV
    gc.collect()

    LISI += lisi_rows(REP, IMG)
    pd.DataFrame(LISI).to_csv(f"{OUT}/lisi.csv", index=False)
    tick(f"rep {REP} LISI")

    SCR = {}
    for task in ("fine", "coarse"):
        for t in ("Jackson", "Danenberg"):
            SCR[(task, t)] = scorer(CO[t], IMG[t]["test"], task, len(labels_of(task, t)), SEED + 31 + REP + 1000 * (t == "Danenberg"))
    ROWS, PCL, LOG, BOOT = [], [], [], {}
    for task, src in UNITS:
        el = (time.time() - T0) / 60
        proj = EST.get((task, src), 30.0)
        if el + proj > BUDGET_MIN:
            print(f"!!! skipping rep {REP} {task} / train {src}: {el:.0f} + {proj:.0f} min > {BUDGET_MIN:.0f}")
            SKIPPED.append((REP, task, src))
            continue
        t_u = time.time()
        ncls = len(COARSE) if task == "coarse" else len(CO[src]["fine"])
        tests = ["Jackson", "Danenberg"] if task == "coarse" else [src]
        for name in (ARMS if task == "coarse" else FINE_ARMS):
            t0 = time.time()
            mdl, n_train = fit_table(ARMS[name], src, task, ncls, IMG, SEED + REP)
            res = {t: score(SCR[(task, t)], predict_table(mdl, ARMS[name], CO[t], IMG[t]["test"], task)) for t in tests}
            LOG.append(dict(rep=REP, task=task, train=src, model=name, n_train=n_train, seconds=time.time() - t0))
            for t, (pt, bt, f1pt, f1bt, cm) in res.items():
                BOOT[(task, src, t, name)] = (pt, bt)
                S = SCR[(task, t)]
                for m, v in pt.items():
                    lo, hi = ci(bt[m])
                    ROWS.append(dict(rep=REP, task=task, train=src, test=t, model=name, metric=m, value=v, lo=lo, hi=hi,
                                     n_cells=len(S["node"]), n_patients=S["npat"]))
                labs = labels_of(task, t)
                for c in np.flatnonzero(S["present"]):
                    lo, hi = ci(f1bt[:, c])
                    PCL.append(dict(rep=REP, task=task, train=src, test=t, model=name, label=labs[c], support=int(cm[c].sum()),
                                    f1=float(f1pt[c]), lo=lo, hi=hi))
            tick(f"rep {REP} {task} / train {src}: {name} " + ", ".join(f"{t} F1 {r[0]['macro_f1']:.3f}" for t, r in res.items()))
            del mdl
        EST[(task, src)] = (time.time() - t_u) / 60
        pd.DataFrame(LOG).to_csv(f"{OUT}/training_log_rep{REP}.csv", index=False)
    RES = pd.DataFrame(ROWS)
    RES.to_csv(f"{OUT}/scores_rep{REP}.csv", index=False)
    pd.DataFrame(PCL).to_csv(f"{OUT}/perclass_rep{REP}.csv", index=False)
    DIFF = []
    for (task, src, t, name), (pt, bt) in BOOT.items():
        kb = (task, src, t, BASE)
        if name == BASE or kb not in BOOT:
            continue
        for m in ("macro_f1", "bal_acc"):
            lo, hi = ci(bt[m] - BOOT[kb][1][m])
            DIFF.append(dict(rep=REP, task=task, train=src, test=t, model=name, base=BASE, metric=m,
                             diff=pt[m] - BOOT[kb][0][m], lo=lo, hi=hi,
                             call="better" if lo > 0 else "worse" if hi < 0 else "n.s."))
    pd.DataFrame(DIFF).to_csv(f"{OUT}/diffs_rep{REP}.csv", index=False)
    if len(RES):
        print(f"--- replicate {REP}: macro-F1")
        print(RES[RES.metric == "macro_f1"].pivot_table(index=["task", "train", "test"], columns="model", values="value")
              .round(3).to_string())
    EST["rep"] = (time.time() - t_rep) / 60
    tick(f"replicate {REP} done ({EST['rep']:.0f} min)")

print("skipped units:", SKIPPED if SKIPPED else "none")
json.dump({"skipped": [list(map(str, s)) for s in SKIPPED], "reps": REPS, "budget_min": BUDGET_MIN, "smoke": SMOKE,
           "minutes": (time.time() - T0) / 60, "device": dev, "novae": novae.__version__,
           "novae_init": NOVAE_INIT, "novae_fit": NOVAE_FIT, "radius_um": RADIUS, "domain_level": DOMAIN_LEVEL},
          open(f"{OUT}/run_info.json", "w"), indent=1)

## Across replicates, sanity check against `cell-type-graph`, and the pre-registered decisions

In [ ]:
fs = sorted(glob.glob(f"{OUT}/scores_rep*.csv"))
ALL = pd.concat([pd.read_csv(f) for f in fs])
AGG = ALL.groupby(["task", "train", "test", "model", "metric"]).value.agg(["mean", "std", "count"]).reset_index()
AGG.to_csv(f"{OUT}/summary_scores.csv", index=False)
print(AGG[AGG.metric == "macro_f1"].pivot_table(index=["task", "train", "test"], columns="model", values="mean").round(3).to_string())

# Table + nbr is the same model, features, split and seed as cell-type-graph: its macro-F1 should match that run
ref = g("summary_scores.csv")
ref = [p for p in ref if "cell-type-graph" in p]
if ref and not SMOKE:
    R = pd.read_csv(ref[0])
    R = R[(R.metric == "macro_f1") & (R.model == "Table + nbr")].set_index(["task", "train", "test"])["mean"]
    M = AGG[(AGG.metric == "macro_f1") & (AGG.model == BASE)].set_index(["task", "train", "test"])["mean"]
    chk = pd.concat({"cell-type-graph": R, "this run": M}, axis=1).dropna()
    chk["diff"] = chk["this run"] - chk["cell-type-graph"]
    print(chk.round(4).to_string())
    chk.to_csv(f"{OUT}/check_vs_cell_type_graph.csv")
else:
    print("cell-type-graph output not mounted (or smoke run): reproduction check skipped")

DF = pd.concat([pd.read_csv(f) for f in sorted(glob.glob(f"{OUT}/diffs_rep*.csv"))])
f1d = DF[(DF.metric == "macro_f1") & (DF.task == "coarse") & (DF.model == "Table + nbr + Novae [cohort]")]
n_reps = DF.rep.nunique()

def cell(src, tgt):
    return f1d[(f1d.train == src) & (f1d.test == tgt)]

transfer = {"J->D": cell("Jackson", "Danenberg"), "D->J": cell("Danenberg", "Jackson")}
within = {"J->J": cell("Jackson", "Jackson"), "D->D": cell("Danenberg", "Danenberg")}
passes = {k: bool(len(v) == 3 and (v.lo > 0).all() and v["diff"].mean() >= 0.01) for k, v in transfer.items()}
losses = {k: bool((v.hi < 0).sum() >= 2) for k, v in {**transfer, **within}.items()}
hn1 = any(passes[d] and not any(losses[o] for o in losses if o != d) for d in passes)

LS = pd.DataFrame(LISI)
lisi_summary = LS.groupby("space")[["ilisi_median", "clisi_median", "ilisi_mean", "clisi_mean"]].agg(["mean", "min", "max"])
print(lisi_summary.round(3).to_string())
lisi_summary.to_csv(f"{OUT}/lisi_summary.csv")
piv = LS.pivot_table(index=["rep", "draw"], columns="space", values=["ilisi_median", "clisi_mean"])
ilisi_higher = bool((piv["ilisi_median"]["Novae [cohort]-corr"] > piv["ilisi_median"]["Table + nbr"]).all())
clisi_rise = float((piv["clisi_mean"]["Novae [cohort]-corr"] - piv["clisi_mean"]["Table + nbr"]).mean())
hn2 = ilisi_higher and clisi_rise < 0.25

decision = dict(
    n_reps=int(n_reps),
    H_N1=dict(claim=bool(hn1 and n_reps == 3), transfer_pass=passes, loss_ge2_reps=losses,
              mean_gain={k: float(v["diff"].mean()) for k, v in {**transfer, **within}.items()},
              ci_lo={k: [float(x) for x in v.lo] for k, v in transfer.items()}),
    H_N2=dict(claim=bool(hn2), ilisi_higher_all_draws=ilisi_higher, mean_clisi_rise=clisi_rise),
    note="H-N1 needs all 3 replicates; with fewer it is reported but cannot be claimed.")
json.dump(decision, open(f"{OUT}/decision.json", "w"), indent=1)
print(json.dumps(decision, indent=1))